# 🗺️ Vector Data Processing Using Python — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Topic |
|---|-------|
| 0 | Setup & Environment |
| 1 | Reading Vector Data (Shapefiles & GeoJSON) |
| 2 | Exploring & Inspecting a GeoDataFrame |
| 3 | Filtering & Subsetting Spatial Data |
| 4 | Reprojecting for Accurate Analysis |
| 5 | Spatial Operations & Attribute Manipulation |
| 6 | Visualizing with Matplotlib |
| 7 | Multi-Panel County Maps (Automation) |
| 8 | Exporting Results |
| 9 | Census Demographic Data with pygris |
| 10 | Choropleth Mapping with Census Data |
| 11 | Bonus — Spatial Joins & Dissolve |

---

# 0️⃣ Setup & Environment

---

> This notebook uses **Google Colab** as the primary environment.  
> Data: North Dakota TIGER census tract shapefiles and U.S. Census ACS data.

| Library | Purpose |
|---------|----------|
| `geopandas` | Read, process, and write vector geospatial data |
| `shapely` | Geometry operations (buffer, centroid, intersect) |
| `matplotlib` | Static map visualization |
| `numpy` | Numerical operations on attribute data |
| `pygris` | Access U.S. Census TIGER and ACS data |
| `fiona` | Underlying file I/O (used by GeoPandas) |

---

### 💡 Example 0.1 — Mount Drive & Create Folders

In [ ]:
# Mount Google Drive (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import os
data_folder   = 'data'
output_folder = 'output'

for folder in [data_folder, output_folder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created: {folder}')
    else:
        print(f'Already exists: {folder}')

### 💡 Example 0.2 — Import All Required Libraries

In [ ]:
# !pip install geopandas shapely matplotlib numpy pygris   # uncomment if needed

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

print('GeoPandas version :', gpd.__version__)
print('All libraries loaded ✅')
!pwd

# 1️⃣ Reading Vector Data

---

> **GeoPandas** reads virtually any vector format into a `GeoDataFrame` — a table with a special `geometry` column.  
> `gpd.read_file()` auto-detects format using **Fiona / GDAL** under the hood.

| Format | Extension | Notes |
|--------|-----------|-------|
| ESRI Shapefile | `.shp` / `.zip` | Most common; requires multiple sidecar files |
| GeoJSON | `.geojson` | Human-readable; single file |
| GeoPackage | `.gpkg` | Modern, single-file alternative to Shapefile |
| Parquet | `.parquet` | Fast, columnar; great for large datasets |
| KML | `.kml` | Google Earth format |

---

### 💡 Example 1.1 — Read a Zipped Shapefile

In [ ]:
import geopandas as gpd

ShapefilePath = r'data/tl_2025_38_tract.zip'

# gpd.read_file() auto-detects the format and reads via Fiona
StateNd = gpd.read_file(ShapefilePath)

# Display the first 5 rows — note the special 'geometry' column
print(f'Type  : {type(StateNd)}')
print(f'Shape : {StateNd.shape}  (rows × columns)')
print(f'Columns: {StateNd.columns.tolist()}')
StateNd.head(5)

### 💡 Example 1.2 — Read GeoJSON and GeoPackage

In [ ]:
import geopandas as gpd

# Reading a GeoJSON (same function, different format)
# gdf_json = gpd.read_file('data/my_layer.geojson')

# Reading a GeoPackage layer
# gdf_gpkg = gpd.read_file('data/my_data.gpkg', layer='layer_name')

# Reading directly from a URL (e.g. GitHub raw GeoJSON)
url = 'https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson'
try:
    world = gpd.read_file(url)
    print(f'World countries loaded: {len(world)} features')
    world.head(3)
except Exception as e:
    print('URL read skipped (no internet):', e)

### 💡 Example 1.3 — Create a GeoDataFrame from Scratch

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import pandas as pd

# From a list of Shapely geometries
cities = gpd.GeoDataFrame({
    'city'      : ['Fargo', 'Bismarck', 'Grand Forks', 'Minot'],
    'population': [125_000, 74_000, 59_000, 48_000],
    'geometry'  : [
        Point(-96.79, 46.88),
        Point(-100.78, 46.81),
        Point(-97.03, 47.93),
        Point(-101.30, 48.23)
    ]
}, crs='EPSG:4326')

print(cities)
cities.plot(markersize=80, color='red', figsize=(6, 4))
plt.title('Major ND Cities')
plt.show()

# 2️⃣ Exploring & Inspecting a GeoDataFrame

---

> A `GeoDataFrame` inherits all Pandas methods **plus** spatial properties.  
> Use these methods to understand the data before any analysis.

| Method / Property | Description |
|-------------------|-------------|
| `.head(n)` / `.tail(n)` | First / last n rows |
| `.info()` | Column names, dtypes, non-null counts |
| `.describe()` | Descriptive statistics for numeric columns |
| `.crs` | Coordinate Reference System |
| `.geom_type` | Geometry type per feature |
| `.total_bounds` | Spatial extent `[minx, miny, maxx, maxy]` |
| `['col'].nunique()` | Number of unique values in a column |
| `['col'].value_counts()` | Frequency of each value |

---

### 💡 Example 2.1 — Basic Metadata Inspection

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

print('=== Shape ===')
print(f'  Rows    : {StateNd.shape[0]}')
print(f'  Columns : {StateNd.shape[1]}')

print('\n=== CRS ===')
print(f'  {StateNd.crs}')

print('\n=== Geometry Types ===')
print(f'  {set(StateNd.geom_type)}')

print('\n=== Spatial Extent ===')
print(f'  {StateNd.total_bounds}')

print('\n=== Column Data Types ===')
print(StateNd.dtypes)

### 💡 Example 2.2 — Attribute Statistics

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Number of unique counties
n_counties = StateNd['COUNTYFP'].nunique()
print(f'Unique counties   : {n_counties}')

# Number of census tracts
print(f'Total tracts      : {len(StateNd)}')

# Top 5 counties by number of tracts
print('\nTop 5 counties by tract count:')
print(StateNd['COUNTYFP'].value_counts().head())

# Descriptive stats on ALAND (land area in m²)
print('\nALAND (land area m²) stats:')
print(StateNd['ALAND'].describe())

### 💡 Example 2.3 — Detailed DataFrame Info

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Full column info (dtype + non-null count)
StateNd.info()

# Check for any missing values
print('\nMissing values per column:')
print(StateNd.isnull().sum())

# 3️⃣ Filtering & Subsetting Spatial Data

---

> GeoPandas inherits Pandas **boolean indexing** — filter rows by attribute conditions exactly as you would a DataFrame.

| Operation | Code |
|-----------|------|
| Single value | `gdf[gdf['col'] == 'value']` |
| Multiple values | `gdf[gdf['col'].isin(['a','b'])]` |
| Numeric range | `gdf[(gdf['col'] > 100) & (gdf['col'] < 500)]` |
| String contains | `gdf[gdf['col'].str.contains('text')]` |
| Select columns | `gdf[['col1','col2','geometry']]` |
| Reset index | `.reset_index(drop=True)` |

---

### 💡 Example 3.1 — Filter by County Code

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Cass County = COUNTYFP '017'
CassCounty = StateNd[StateNd['COUNTYFP'] == '017'].copy()
print(f'Cass County tracts : {len(CassCounty)}')
CassCounty.head(3)

### 💡 Example 3.2 — Filter by Multiple Conditions

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Select multiple counties
selected_counties = ['017', '035', '057']
multi_county = StateNd[StateNd['COUNTYFP'].isin(selected_counties)].copy()
print(f'Selected counties row count: {len(multi_county)}')

# Filter by land area: tracts larger than 500 km²
large_tracts = StateNd[StateNd['ALAND'] > 500_000_000].copy()
print(f'Tracts > 500 km²: {len(large_tracts)}')

# Combine: large tracts in Cass County
cass_large = StateNd[
    (StateNd['COUNTYFP'] == '017') & (StateNd['ALAND'] > 50_000_000)
].copy()
print(f'Large tracts in Cass County: {len(cass_large)}')

### 💡 Example 3.3 — Select Specific Columns & Reset Index

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Keep only relevant columns
subset = StateNd[['GEOID', 'COUNTYFP', 'TRACTCE', 'ALAND', 'AWATER', 'geometry']].copy()
print('Subset columns:', subset.columns.tolist())

# Filter and reset index so rows are numbered from 0
cass = subset[subset['COUNTYFP'] == '017'].reset_index(drop=True)
print(f'\nCass County subset, index reset:')
print(cass.head(3))

# 4️⃣ Reprojecting for Accurate Analysis

---

> **CRS matters**: Geographic CRS (degrees) → distances and areas are meaningless.  
> Always reproject to a **projected CRS** (metres) before spatial calculations.

| CRS | EPSG | Units | Best For |
|-----|------|-------|----------|
| WGS 84 | 4326 | Degrees | Storage, web mapping |
| NAD83 | 4269 | Degrees | US Census default |
| US Albers (CONUS) | 5070 | Metres | Area calculations (CONUS) |
| UTM Zone 14N | 32614 | Metres | Local / state accuracy |
| Web Mercator | 3857 | Metres (distorted) | Tile-based web maps |

---

### 💡 Example 4.1 — Reproject to EPSG:5070

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

print('Original CRS :', StateNd.crs)

# Reproject to US Albers Equal Area (metres)
StateNdProj = StateNd.to_crs(epsg=5070)

print('Projected CRS:', StateNdProj.crs)
print('\nOriginal bounds  :', StateNd.total_bounds.round(4))
print('Projected bounds :', StateNdProj.total_bounds.round(0))

### 💡 Example 4.2 — Why Projection Matters for Area

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Area in geographic CRS (degrees² — meaningless)
area_deg = StateNd.geometry.area

# Area after projecting to metres
StateNdProj = StateNd.to_crs(epsg=5070)
area_m2 = StateNdProj.geometry.area

print('Area in degrees² (first 3):' , area_deg.head(3).values)
print('Area in m²       (first 3):' , area_m2.head(3).values.round(0))
print('Area in km²      (first 3):' , (area_m2.head(3)/1e6).values.round(2))

# 5️⃣ Spatial Operations & Attribute Manipulation

---

> GeoPandas exposes **Shapely geometry methods** on the entire GeoSeries column at once.

| Operation | Code | Returns |
|-----------|------|---------|
| Area | `gdf.geometry.area` | GeoSeries of floats |
| Length / Perimeter | `gdf.geometry.length` | GeoSeries of floats |
| Centroid | `gdf.geometry.centroid` | GeoSeries of Points |
| Bounding box | `gdf.geometry.envelope` | GeoSeries of Polygons |
| Buffer | `gdf.geometry.buffer(dist)` | GeoSeries of Polygons |
| Convex hull | `gdf.geometry.convex_hull` | GeoSeries of Polygons |
| Simplify | `gdf.geometry.simplify(tol)` | GeoSeries |
| Is valid | `gdf.geometry.is_valid` | GeoSeries of bools |

---

### 💡 Example 5.1 — Calculate Area & Centroid

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')
StateNdProj = StateNd.to_crs(epsg=5070)

# Area in m² and km²
StateNdProj['AreaM2']  = StateNdProj.geometry.area
StateNdProj['AreaKm2'] = StateNdProj['AreaM2'] / 1_000_000

# Centroid (also in projected metres)
StateNdProj['Centroid'] = StateNdProj.geometry.centroid

# Perimeter
StateNdProj['PerimM'] = StateNdProj.geometry.length

print(StateNdProj[['GEOID','COUNTYFP','AreaKm2','PerimM']].head())
print(f'\nLargest tract : {StateNdProj["AreaKm2"].max():.1f} km²')
print(f'Smallest tract: {StateNdProj["AreaKm2"].min():.2f} km²')

### 💡 Example 5.2 — Buffer, Envelope & Convex Hull

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')
cass    = StateNd[StateNd['COUNTYFP']=='017'].to_crs(5070)

# Dissolve all Cass County tracts into one polygon
cass_union   = cass.dissolve()

# Buffer: expand boundary by 5 km
cass_buffer  = cass_union.copy()
cass_buffer['geometry'] = cass_union.geometry.buffer(5000)

# Convex hull: smallest convex polygon containing all features
cass_hull = cass_union.copy()
cass_hull['geometry'] = cass_union.geometry.convex_hull

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, gdf, title in zip(axes,
    [cass_union, cass_buffer, cass_hull],
    ['Dissolved Union', '5 km Buffer', 'Convex Hull']):
    gdf.plot(ax=ax, facecolor='lightblue', edgecolor='navy')
    ax.set_title(title); ax.set_axis_off()
plt.tight_layout()
plt.show()

### 💡 Example 5.3 — Validate & Fix Geometries

In [ ]:
import geopandas as gpd

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip')

# Check for invalid geometries
invalid = StateNd[~StateNd.geometry.is_valid]
print(f'Invalid geometries: {len(invalid)}')

# Fix invalid geometries with buffer(0) trick
StateNd['geometry'] = StateNd.geometry.buffer(0)
print(f'Invalid after fix : {(~StateNd.geometry.is_valid).sum()}')

# Check for empty geometries
empty = StateNd[StateNd.geometry.is_empty]
print(f'Empty geometries  : {len(empty)}')

# 6️⃣ Visualizing with Matplotlib

---

> `gdf.plot()` is a GeoPandas wrapper around Matplotlib. Every parameter you know from Matplotlib applies.

| Parameter | Description |
|-----------|-------------|
| `column` | Attribute to colour features by |
| `cmap` | Colormap (e.g. `'RdYlBu'`, `'viridis'`, `'cividis'`) |
| `legend=True` | Show colorbar |
| `facecolor` | Fill colour for polygons |
| `edgecolor` | Border colour |
| `linewidth` | Border thickness |
| `alpha` | Transparency (0–1) |
| `missing_kwds` | Style for features with no data |
| `scheme` | Classification scheme (`'quantiles'`, `'equal_interval'`) |

---

### 💡 Example 6.1 — Basic Map & Choropleth

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Basic outline map
StateNdProj.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.4)
ax1.set_title('North Dakota Census Tracts (outline)')
ax1.set_axis_off()

# Choropleth by area
StateNdProj.plot(
    ax=ax2, column='AreaKm2',
    cmap='RdYlBu', legend=True,
    legend_kwds={'label': 'Area (km²)', 'shrink': 0.5}
)
ax2.set_title('Tract Area Choropleth (km²)')
ax2.set_axis_off()

plt.tight_layout()
plt.show()

### 💡 Example 6.2 — Styled Map with Custom Colours

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNd = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNd['AreaKm2'] = StateNd.geometry.area / 1e6

fig, ax = plt.subplots(figsize=(12, 9))

StateNd.plot(
    ax=ax,
    column='AreaKm2',
    cmap='cividis',          # colorblind-friendly
    scheme='quantiles',      # classify into equal quantile bins
    k=5,                     # 5 classes
    legend=True,
    legend_kwds={'title': 'Area (km²)', 'loc': 'lower right'},
    edgecolor='white',
    linewidth=0.3,
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

ax.set_title('North Dakota Census Tract Area\n(5 Quantile Classes)',
             fontsize=14, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('output/nd_area_choropleth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/nd_area_choropleth.png')

# 7️⃣ Multi-Panel County Maps (Automation)

---

> Use Python loops + `plt.subplots()` to generate small multiples — one map per county — automatically.

| Concept | Code |
|---------|------|
| Get unique values | `gdf['col'].unique()` |
| Create subplot grid | `fig, axes = plt.subplots(nrows, ncols, figsize=...)` |
| Flatten axes | `axes.flatten()` |
| Hide unused axes | `ax.set_visible(False)` |
| Tight layout | `plt.tight_layout()` |

---

### 💡 Example 7.1 — Plot First 20 Counties in a Grid

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaM2'] = StateNdProj.geometry.area

counties = StateNdProj['COUNTYFP'].unique()
print(f'Total unique counties: {len(counties)}')

# 5×4 grid = 20 panels
fig, axes = plt.subplots(5, 4, figsize=(24, 15))
axes_flat = axes.flatten()

for ax, fips in zip(axes_flat, counties[:20]):
    county_gdf = StateNdProj[StateNdProj['COUNTYFP'] == fips]
    county_gdf.plot(
        column='AreaM2', ax=ax,
        cmap='RdYlBu', legend=False
    )
    ax.set_title(f'County {fips}', fontsize=9)
    ax.set_axis_off()

# Hide any unused subplots
for ax in axes_flat[len(counties[:20]):]:
    ax.set_visible(False)

plt.suptitle('North Dakota Counties — Tract Area (first 20)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('output/county_grid.png', dpi=120, bbox_inches='tight')
plt.show()

### 💡 Example 7.2 — Add County Statistics to Each Panel

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

counties = StateNdProj['COUNTYFP'].unique()[:6]   # first 6 for demo

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, fips in zip(axes.flatten(), counties):
    county_gdf = StateNdProj[StateNdProj['COUNTYFP'] == fips]
    county_gdf.plot(column='AreaKm2', ax=ax, cmap='Blues',
                    edgecolor='black', linewidth=0.5)
    n      = len(county_gdf)
    total  = county_gdf['AreaKm2'].sum()
    ax.set_title(f'County {fips}\n{n} tracts | {total:.0f} km²', fontsize=10)
    ax.set_axis_off()

plt.suptitle('ND County Maps with Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 8️⃣ Exporting Results

---

> GeoPandas can write to many formats. Choose the right one for your workflow.

| Format | Method | Notes |
|--------|--------|-------|
| Shapefile | `gdf.to_file('out.shp')` | Multiple sidecar files; widely supported |
| GeoJSON | `gdf.to_file('out.geojson', driver='GeoJSON')` | Single file; human-readable |
| GeoPackage | `gdf.to_file('out.gpkg', driver='GPKG')` | Single file; supports layers |
| Parquet | `gdf.to_parquet('out.parquet')` | Fast, columnar; large data |
| CSV | `df.to_csv('out.csv')` | Drop geometry or export as WKT |

---

### 💡 Example 8.1 — Export to Multiple Formats

In [ ]:
import geopandas as gpd
import warnings, os

StateNd     = gpd.read_file(r'data/tl_2025_38_tract.zip')
StateNdProj = StateNd.to_crs(5070)
StateNdProj['AreaKm2']  = StateNdProj.geometry.area / 1e6
StateNdProj['PerimM']   = StateNdProj.geometry.length
StateNdProj['Centroid_WKT'] = StateNdProj.geometry.centroid.to_wkt()

# Drop extra geometry column before saving formats that don't support two
save_gdf = StateNdProj.drop(columns=['Centroid_WKT'] if 'Centroid_WKT' not in StateNdProj.columns else [])

# Shapefile
StateNdProj.to_file('output/nd_tracts_proj.shp')

# GeoPackage
warnings.filterwarnings('ignore')
StateNdProj.to_file('output/nd_tracts_proj.gpkg', driver='GPKG', layer='tracts')

# GeoJSON
StateNdProj.to_file('output/nd_tracts_proj.geojson', driver='GeoJSON')

# Parquet (fastest for large data)
StateNdProj.to_parquet('output/nd_tracts_proj.parquet')

# CSV with WKT geometry
csv_df = StateNdProj.copy()
csv_df['geometry'] = csv_df.geometry.to_wkt()
csv_df.drop(columns=[c for c in csv_df.columns if 'Centroid' in c], errors='ignore').to_csv('output/nd_tracts_proj.csv', index=False)

for fname in ['nd_tracts_proj.shp','nd_tracts_proj.gpkg','nd_tracts_proj.geojson',
              'nd_tracts_proj.parquet','nd_tracts_proj.csv']:
    path = f'output/{fname}'
    if os.path.exists(path):
        print(f'{fname:<35} {os.path.getsize(path)//1024:>6} KB')

# 9️⃣ Census Demographic Data with pygris

---

> **pygris** provides a Pythonic interface to the U.S. Census Bureau API.  
> `get_census()` retrieves ACS (American Community Survey) variables for any geography.

| Parameter | Description |
|-----------|-------------|
| `dataset` | Census dataset (e.g. `'acs/acs5/profile'`) |
| `variables` | ACS variable ID (e.g. `'DP02_0067PE'`) |
| `year` | ACS year (e.g. `2021`) |
| `params` | Dict: geography scope (state, county, tract) |
| `return_geoid` | Return GEOID column for joining to spatial data |

> **Useful ACS links:**  
> 🔗 [Census API datasets](https://api.census.gov/data.html)  
> 🔗 [ACS 5-yr Data Profile variables](https://api.census.gov/data/2021/acs/acs5/profile/variables.html)

---

### 💡 Example 9.1 — Install pygris & Retrieve ACS Data

In [ ]:
# !pip install pygris   # uncomment if needed
from pygris.data import get_census

# DP02_0067PE = % Population 25+ with High School Diploma or Higher
nd_education = get_census(
    dataset    = 'acs/acs5/profile',
    variables  = 'DP02_0067PE',
    year       = 2021,
    params     = {'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes  = True,
    return_geoid  = True
)

print(f'Rows returned : {len(nd_education)}')
print(f'Columns       : {nd_education.columns.tolist()}')
nd_education.head()

### 💡 Example 9.2 — Retrieve Multiple Variables

In [ ]:
from pygris.data import get_census

# Retrieve multiple ACS variables at once
nd_multi = get_census(
    dataset   = 'acs/acs5/profile',
    variables = ['DP02_0067PE',   # % HS Graduate+
                 'DP03_0062E',    # Median household income
                 'DP05_0001E'],   # Total population
    year      = 2021,
    params    = {'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes = True,
    return_geoid = True
)

nd_multi.rename(columns={
    'DP02_0067PE': 'pct_hs_grad',
    'DP03_0062E' : 'median_income',
    'DP05_0001E' : 'total_pop'
}, inplace=True)

print(nd_multi[['GEOID','pct_hs_grad','median_income','total_pop']].head())

# 🔟 Choropleth Mapping with Census Data

---

> Join ACS tabular data to the spatial GeoDataFrame using **GEOID** as the common key,  
> then visualize with a polished choropleth map.

| Step | Code |
|------|------|
| Merge | `gdf.merge(df, on='GEOID', how='inner')` |
| Basic plot | `merged.plot(column='var', legend=True)` |
| Styled plot | custom `figsize`, `cmap`, `edgecolor`, `legend_kwds` |
| Colorblind palette | `'cividis'`, `'viridis'`, `'RdYlBu'` |

---

### 💡 Example 10.1 — Merge ACS Data & Basic Choropleth

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

# Load spatial data
StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# Load ACS education variable
nd_edu = get_census(
    dataset='acs/acs5/profile', variables='DP02_0067PE', year=2021,
    params={'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes=True, return_geoid=True
)

# Merge on GEOID
ND_merged = StateNdProj.merge(nd_edu, how='inner', on='GEOID')
print(f'Merged rows: {len(ND_merged)}')

# Quick choropleth
ND_merged.plot(
    column='DP02_0067PE', legend=True,
    legend_kwds={'shrink': 0.4},
    figsize=(10, 8)
)
plt.title('% High School Graduate (Population 25+) — North Dakota')
plt.show()

### 💡 Example 10.2 — Polished Publication-Ready Map

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
nd_edu      = get_census(dataset='acs/acs5/profile', variables='DP02_0067PE', year=2021,
                         params={'for':'tract:*','in':'state:38'},
                         guess_dtypes=True, return_geoid=True)
ND_merged   = StateNdProj.merge(nd_edu, how='inner', on='GEOID')

fig, ax = plt.subplots(figsize=(14, 10))

ND_merged.plot(
    ax=ax,
    column='DP02_0067PE',
    cmap='cividis',             # colorblind-friendly
    linewidth=0.5,
    edgecolor='white',
    legend=True,
    legend_kwds={'shrink': 0.5, 'label': '% HS Graduate (Pop 25+)'},
    missing_kwds={'color': 'lightgrey', 'label': 'No data'}
)

ax.set_title(
    '% Population 25+ with High School Diploma or Higher\nNorth Dakota — ACS 2021 (5-Year)',
    fontsize=15, fontweight='bold', pad=15
)
ax.annotate('Source: U.S. Census Bureau, ACS 2021',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=9, color='grey')
ax.axis('off')
plt.tight_layout()
plt.savefig('output/nd_education_map.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/nd_education_map.png')

### 💡 Example 10.3 — Internet Access Map (Exercise Variable)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pygris.data import get_census

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# DP02_0152PE = % Households with computer and internet access
nd_internet = get_census(
    dataset='acs/acs5/profile', variables='DP02_0152PE', year=2021,
    params={'for': 'tract:*', 'in': 'state:38'},
    guess_dtypes=True, return_geoid=True
)

ND_merged2 = StateNdProj.merge(nd_internet, how='inner', on='GEOID')

fig, ax = plt.subplots(figsize=(14, 10))
ND_merged2.plot(
    ax=ax, column='DP02_0152PE',
    cmap='YlGnBu', linewidth=0.4, edgecolor='white',
    legend=True,
    legend_kwds={'shrink': 0.5, 'label': '% Households — Computers & Internet'},
    missing_kwds={'color': 'lightgrey'}
)
ax.set_title('% Households with Computer & Internet Access\nNorth Dakota — ACS 2021 (5-Year)',
             fontsize=15, fontweight='bold')
ax.annotate('Source: U.S. Census Bureau, ACS 2021',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=9, color='grey')
ax.axis('off')
plt.tight_layout()
plt.savefig('output/nd_internet_map.png', dpi=200, bbox_inches='tight')
plt.show()

# 1️⃣1️⃣ Bonus — Spatial Joins & Dissolve

---

> **Spatial join** assigns attributes from one layer to another based on spatial relationships.  
> **Dissolve** merges geometries sharing a common attribute into one polygon.

| Operation | Code | Predicate options |
|-----------|------|-------------------|
| Spatial join | `gpd.sjoin(left, right, how='left', predicate='within')` | `within`, `intersects`, `contains` |
| Dissolve | `gdf.dissolve(by='column', aggfunc='sum')` | `sum`, `mean`, `count`, `first` |
| Union all | `gdf.dissolve()` | (no by= → one polygon) |

---

### 💡 Example 11.1 — Dissolve Tracts into Counties

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)
StateNdProj['AreaKm2'] = StateNdProj.geometry.area / 1e6

# Dissolve all tracts into counties — summing ALAND and AreaKm2
counties = StateNdProj.dissolve(
    by='COUNTYFP',
    aggfunc={'ALAND': 'sum', 'AreaKm2': 'sum', 'TRACTCE': 'count'}
).rename(columns={'TRACTCE': 'num_tracts'}).reset_index()

print(f'Counties dissolved: {len(counties)}')
print(counties[['COUNTYFP','num_tracts','AreaKm2']].head())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
StateNdProj.plot(ax=ax1, facecolor='none', edgecolor='grey', linewidth=0.3)
ax1.set_title('Census Tracts'); ax1.set_axis_off()

counties.plot(ax=ax2, column='num_tracts', cmap='YlOrRd', legend=True,
              edgecolor='black', linewidth=0.5)
ax2.set_title('Counties — Tract Count'); ax2.set_axis_off()

plt.tight_layout()
plt.show()

### 💡 Example 11.2 — Spatial Join (Points in Polygons)

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import matplotlib.pyplot as plt

StateNdProj = gpd.read_file(r'data/tl_2025_38_tract.zip').to_crs(5070)

# Create 50 random points within ND bounding box
rng = np.random.default_rng(42)
minx, miny, maxx, maxy = StateNdProj.total_bounds
rand_pts = gpd.GeoDataFrame(
    {'id': range(50)},
    geometry=[Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy)) for _ in range(50)],
    crs=StateNdProj.crs
)

# Spatial join: find which tract each point falls in
joined = gpd.sjoin(rand_pts, StateNdProj[['GEOID','COUNTYFP','geometry']],
                   how='left', predicate='within')

print(f'Points with a matching tract: {joined["GEOID"].notna().sum()}')
print(joined[['id','GEOID','COUNTYFP']].head())

fig, ax = plt.subplots(figsize=(10, 8))
StateNdProj.plot(ax=ax, facecolor='lightyellow', edgecolor='grey', linewidth=0.3)
rand_pts.plot(ax=ax, color='red', markersize=20, zorder=5)
ax.set_title('Random Points Spatially Joined to ND Tracts')
ax.set_axis_off()
plt.show()

---

# 🎉 Congratulations — All 12 Sections Complete!

---

| ✅ | Section |
|----|----------|
| ✅ | 0 — Setup & Environment |
| ✅ | 1 — Reading Vector Data |
| ✅ | 2 — Exploring & Inspecting a GeoDataFrame |
| ✅ | 3 — Filtering & Subsetting Spatial Data |
| ✅ | 4 — Reprojecting for Accurate Analysis |
| ✅ | 5 — Spatial Operations & Attribute Manipulation |
| ✅ | 6 — Visualizing with Matplotlib |
| ✅ | 7 — Multi-Panel County Maps |
| ✅ | 8 — Exporting Results |
| ✅ | 9 — Census Demographic Data with pygris |
| ✅ | 10 — Choropleth Mapping with Census Data |
| ✅ | 11 — Bonus: Spatial Joins & Dissolve |

---

## 🚀 Next Steps

| Topic | Libraries |
|-------|-----------|
| Network analysis | `osmnx`, `networkx` |
| Spatial statistics | `pysal`, `esda`, `libpysal` |
| Interactive maps | `folium`, `leafmap`, `kepler.gl` |
| 3D vector viz | `pydeck`, `plotly` |
| Raster + Vector combined | `rasterio`, `rasterstats` |

---

### Happy Mapping! 🗺️💻

*Every polygon has a story — GeoPandas helps you tell it.*

---